# Hugging Face 图片物体识别 Demo

本示例使用本地图片路径作为输入，识别图片中的常见物体，并在图片上绘制边框与标签。

In [ ]:
# 可选：镜像与超时配置（请在 import transformers 前运行）
import os

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")

print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))
print("HF_HUB_DOWNLOAD_TIMEOUT =", os.environ.get("HF_HUB_DOWNLOAD_TIMEOUT"))

In [ ]:
# 依赖导入（若缺包会自动安装）
from pathlib import Path
import sys
import subprocess


def ensure_package(import_name: str, pip_name: str | None = None):
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        pkg = pip_name or import_name
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])


ensure_package("PIL", "pillow")
ensure_package("matplotlib")
ensure_package("timm")
ensure_package("gradio")

from transformers import pipeline, __version__ as transformers_version
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import gradio as gr

print("transformers version:", transformers_version)

In [ ]:
# 加载通用目标检测模型（首次运行会下载）
detector = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50",
)
print("detector loaded")

In [ ]:
# 参数区：请修改为你的本地图片路径
image_path = r"D:\\code\\claude_code\\hugging_face\\sample.jpg"
score_threshold = 0.6
save_output = True
output_path = r"D:\\code\\claude_code\\hugging_face\\output_annotated.jpg"

img_path = Path(image_path)
if not img_path.exists():
    raise FileNotFoundError(f"图片不存在，请检查路径: {img_path}")

raw_results = detector(str(img_path))
results = [r for r in raw_results if float(r["score"]) >= score_threshold]

print(f"total detections: {len(raw_results)}")
print(f"detections after threshold({score_threshold}): {len(results)}")
for i, r in enumerate(results, 1):
    print(i, {
        "label": r["label"],
        "score": round(float(r["score"]), 4),
        "box": r["box"],
    })

In [ ]:
# 可视化-步骤1：读取图片并检查输入变量
assert "results" in globals(), "请先运行上一个检测单元，确保有 results"
assert "image_path" in globals(), "请先设置 image_path"

image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)
font = ImageFont.load_default()

print("image size:", image.size)
print("results count:", len(results))

In [ ]:
# 可视化-步骤2：逐个画框并打印前几个标签
for i, r in enumerate(results, 1):
    box = r["box"]
    xmin, ymin = int(box["xmin"]), int(box["ymin"])
    xmax, ymax = int(box["xmax"]), int(box["ymax"])
    label = r["label"]
    score = float(r["score"])
    text = f"{label}: {score:.2f}"

    draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
    draw.text((xmin, max(0, ymin - 12)), text, fill="red", font=font)

    if i <= 3:
        print(f"drawn {i}: {text} @ {(xmin, ymin, xmax, ymax)}")

print("all boxes drawn")

In [ ]:
# 可视化-步骤3：显示与保存
plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

if save_output:
    image.save(output_path)
    print("annotated image saved to:", output_path)

## 运行说明与常见问题

1. 首次运行模型会下载，时间取决于网络，镜像环境通常更稳定。
2. 如果出现下载中断（例如 RemoteProtocolError），重试当前单元，或切换网络后重试。
3. 如果报 FileNotFoundError，请确认 `image_path` 是否正确。
4. 如果内存紧张，可提高 `score_threshold` 或先用更小尺寸图片测试。
5. 若想检测更少误报，可将 `score_threshold` 从 0.6 调高到 0.7 或 0.8。

## Gradio 交互式 Demo（上传图片检测）

运行下面两个代码格后，会出现本地访问地址；如需外网分享可设置 `share=True`。

In [ ]:
# Gradio App：上传图片 -> 目标检测 -> 返回标注图与结果表
# 先确保上面的 detector 已成功加载。

def detect_and_draw(image, score_threshold=0.6):
    if image is None:
        return None, []

    img = image.convert("RGB")
    raw_results = detector(img)
    results = [r for r in raw_results if float(r["score"]) >= float(score_threshold)]

    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()

    pretty = []
    for r in results:
        box = r["box"]
        xmin, ymin = int(box["xmin"]), int(box["ymin"])
        xmax, ymax = int(box["xmax"]), int(box["ymax"])
        label = r["label"]
        score = float(r["score"])

        draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
        draw.text((xmin, max(0, ymin - 12)), f"{label}: {score:.2f}", fill="red", font=font)

        pretty.append(
            {
                "label": label,
                "score": round(score, 4),
                "box": {"xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax},
            }
        )

    return img, pretty


demo = gr.Interface(
    fn=detect_and_draw,
    inputs=[
        gr.Image(type="pil", label="上传图片"),
        gr.Slider(0.1, 0.95, value=0.6, step=0.05, label="置信度阈值"),
    ],
    outputs=[
        gr.Image(type="pil", label="标注结果"),
        gr.JSON(label="检测结果"),
    ],
    title="Hugging Face 目标检测 Demo",
    description="模型: facebook/detr-resnet-50。上传图片后返回打框结果和结构化标签。",
    flagging_mode="never",
)

# 生成公网临时链接（他人可访问）
# prevent_thread_lock=True: 启动后不阻塞 notebook 后续单元
demo.launch(share=True, prevent_thread_lock=True)